In [1]:
from fuzzycar import Car
from pynq import Overlay , MMIO

In [2]:
ol = Overlay("car.bit", ignore_version=True)
print("✔ Overlay 'car.bit' loaded successfully.\n")

✔ Overlay 'car.bit' loaded successfully.



In [3]:
car = Car(ol)

Configuring SPI controller...


In [4]:
gpio = MMIO(ol.axi_gpio_0.mmio.base_addr, 0x10000 , debug=False)

In [5]:
gpio.write(0x00,0xff)

In [6]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import time
from IPython.display import clear_output
print("Fuzzy imported libraries complete!")

Fuzzy imported libraries complete!


In [7]:
# Define fuzzy variables
Error=ctrl.Antecedent(np.arange(-1,1.01,0.01),'Error')
Delta_Error=ctrl.Antecedent(np.arange(-1,1.01,0.01),'Delta_Error')
Speed=ctrl.Consequent(np.arange(1660,1679,1),'Speed')
print("Fuzzy variables defined")

Fuzzy variables defined


In [8]:
# Membership functions

# Membership function for Error
Error['Negative']=fuzz.trapmf(Error.universe,[-1, -1, -0.15, -0.05])
Error['None']=fuzz.trimf(Error.universe,[-0.2, 0, 0.2])
Error['Positive']=fuzz.trapmf(Error.universe,[0.05, 0.15, 1, 1])
print("Error MFs defined")

# Membership function for Delta_Error
Delta_Error['Decreasing']=fuzz.trapmf(Delta_Error.universe,[-1, -1, -0.15, -0.05])
Delta_Error['Stable']=fuzz.trimf(Delta_Error.universe,[-0.2, 0, 0.2])
Delta_Error['Increasing']=fuzz.trapmf(Delta_Error.universe,[0.05, 0.15, 1, 1])
print("Delta_Error MFs defined")

#Membership functions for Speed
Speed['Idle']=fuzz.trapmf(Speed.universe,[1660, 1660, 1662, 1662])
Speed['Slow']=fuzz.trimf(Speed.universe,[1660, 1669, 1672])
Speed['Medium']=fuzz.trimf(Speed.universe,[1669, 1672, 1675])
Speed['Fast']=fuzz.trimf(Speed.universe,[1672, 1675, 1678])
print("Speed MFs defined")

Error MFs defined
Delta_Error MFs defined
Speed MFs defined


In [9]:
# Define Rules

rule1=ctrl.Rule(Error['Negative'] & Delta_Error['Decreasing'], Speed['Idle'])
rule2=ctrl.Rule(Error['Negative'] & Delta_Error['Stable'],     Speed['Slow'])
rule3=ctrl.Rule(Error['Negative'] & Delta_Error['Increasing'], Speed['Slow'])
rule4=ctrl.Rule(Error['None']     & Delta_Error['Decreasing'], Speed['Slow'])
rule5=ctrl.Rule(Error['None']     & Delta_Error['Stable'],     Speed['Medium'])
rule6=ctrl.Rule(Error['None']     & Delta_Error['Increasing'], Speed['Medium'])
rule7=ctrl.Rule(Error['Positive'] & Delta_Error['Decreasing'], Speed['Medium'])
rule8=ctrl.Rule(Error['Positive'] & Delta_Error['Stable'],     Speed['Medium'])
rule9=ctrl.Rule(Error['Positive'] & Delta_Error['Increasing'], Speed['Fast'])
print("Fuzzy rules defined")

Fuzzy rules defined


In [11]:
# Control system and simulation
speed_ctrl=ctrl.ControlSystem([rule1, rule2, rule3,
                               rule4, rule5, rule6,
                               rule7, rule8, rule9])
speed_simulation=ctrl.ControlSystemSimulation(speed_ctrl)
print("Control System Defined")

Control System Defined


In [13]:
# For testing inputs

speed_simulation.input['Error'] = -0.4
speed_simulation.input['Delta_Error'] = 0.1
speed_simulation.compute()
print(speed_simulation.output['Speed'])

1666.6666666666667
